# Laboratorium 9 - Sieci Rekurencyjne

Na tym laboratorium zapoznamy się z sieciami rekurencyjnymi - jedną z architektur dedykowanych danym sekwencyjnym. Dla danych sekwencyjnych, jednym z naturalnych podejść do modelowania jest założenie, że przetwarzając n-ty krok, możemy udostępnić modelowi pewną reprezentację historii/pamięci/stanu która w kompaktowy sposób przechowuje informację o "wszystkim co działo się wcześniej". W ten sposób zadania upraszcza sobie często np. modelowanie procesami stochastycznymi (założenie własności Markowa w modelu). W kontekście sieci neuronowych, odpowiednikiem takiego podejścia jest właśnie warstwa rekurencyjna: taka, która w n-tym kroku przetwarza n-te wejście i reprezentacje ukrytą z kroku n-1 (historię/stan/pamięć).

Oczywiście z góry warstwo wspomnieć o fakcie, że architektury rekurencyjne obecnie nie są pierwszym wyborem, w pracach state=of-the-art dominuje mechanizm uwagi. Transformery to obecnie podstawa wszystkiego, co kojarzone z sztuczną inteligencją - w szczególności wszelkiej maści "czatów" opartych o LLM (Large Language Models). Ale ich historia to w pierwszej kolejności dodanie mechanizmu uwagi do sieci rekurencyjnych, a dopiero potem spostrzeżenie, że po dodaniu tego mechanizmu, rekurencja nie jest już w zasadzie konieczna (bardzo znana publikacja *Attention Is All You Need*).

 Na tych laboratoriach przyjrzyjmy się warstwom LSTM - najpopularniejszej wersji architektury czysto rekurencyjnej.

# Zbiory danych

Przeprowadzimy test na dwóch zbiorach danych: klasyczny zbiór do analizy sentymentu tekstów (recenzje IMDB), oraz zbiór audio - rozpoznawanie owadów po wydawanych rzez nich odgłosach.

In [1]:
from tensorflow.keras.datasets import imdb

(x_train, y_train), (x_test, y_test) = imdb.load_data(
    path='imdb.npz',
    num_words=None,
    skip_top=0,
    maxlen=None,
    seed=113,
    start_char=1,
    oov_char=2,
    index_from=3
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 22665, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 21631, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 19193, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 10311, 8, 4, 107, 117, 5952, 15, 256, 4, 31050, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 12118, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [3]:
!pip install aeon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.3 MB/s eta 0:00:00


In [4]:
from aeon.datasets import load_classification
X, y = load_classification("InsectWingbeat", extract_path="insect_data")

/tmp/ipykernel_12039/3839978134.py:2: FutureWarning: Call to deprecated function (or staticmethod) load_classification. (load_classification parameters load_equal_length and load_no_missing will default to False in version 1.5.0) -- Deprecated since version 1.4.0.
  X, y = load_classification("InsectWingbeat", extract_path="insect_data")


In [5]:
print(X[0])

[[-5.2600e-04 -6.8000e-04 -1.0782e-02 ...  1.9104e-02 -1.8969e-02
  -9.8000e-04]
 [ 4.2600e-04  2.9300e-04  1.3112e-02 ... -9.1000e-05 -3.2437e-02
  -2.5000e-05]
 [-4.9000e-04 -4.4300e-04 -1.2667e-02 ... -9.7161e-01 -2.8844e-02
   9.1500e-04]
 ...
 [ 6.5200e-04  5.2600e-04 -2.0744e-02 ...  2.0344e-02 -4.3800e-03
   7.9350e-03]
 [-8.1800e-04  2.4800e-04  2.6454e-02 ...  2.0116e-02 -3.5250e-03
  -2.4500e-04]
 [-8.1800e-04  2.4800e-04  2.6454e-02 ...  2.0116e-02 -3.5250e-03
  -2.4500e-04]]


# Wczytywanie danych sekwencyjnych

Dla danych sekwencyjnych, przy przetwarzaniu w batchu pojawia się nowa komplikacja: batch musi być możliwy do "spakowania" w tensorze `batch_size x vector_dimension x sequence_length `, podczas gdy sekwencje w zbiorze uczącym nie muszą być jednakowego rozmiaru. Rozwiązaniem jest padding: dopełnianie tensorów zerami do stałej długości.

W problemach klasyfikacyjnych to powoduje jednak nowy problem: chcemy uzyskać reprezentację do klasyfikacji na końcu właściwej sekwencji, nie po przetworzeniu kilu (nastu/dziesięciu/set) dodatkowych wektorów zer. Implementacje warstw rekurencyjnych w torchu oferują nam rozwiązanie w postaci obiektów PackedSequence. Odpowiednimi funkcjami możemy "spakować" zarówno listę sekencji, jak i tensor już wypadowanych danych z podanymi osobno długościami sekwencji. Podanie takiej paczki na wejści warstwy rekurencyjnej gwarantuje, że warstwa zwróci nam swoją reprezentację na poziomie **ostatniego elementu właściwej sekwencji** (nie biorąc pod uwagę paddingu).

In [6]:
import torch

sequences = [[0,1,2,3],
             [0,1],
             [0]]

packed_sequences = torch.nn.utils.rnn.pack_sequence([torch.tensor(seq) for seq in sequences])
print(packed_sequences)

padded_sequences = torch.nn.utils.rnn.pad_sequence([torch.tensor(seq) for seq in sequences])
print(padded_sequences)

packed_padded_sequences = torch.nn.utils.rnn.pack_padded_sequence(padded_sequences, [len(s) for s in sequences])
print(packed_padded_sequences)

PackedSequence(data=tensor([0, 0, 0, 1, 1, 2, 3]), batch_sizes=tensor([3, 2, 1, 1]), sorted_indices=None, unsorted_indices=None)
tensor([[0, 0, 0],
        [1, 1, 0],
        [2, 0, 0],
        [3, 0, 0]])
PackedSequence(data=tensor([0, 0, 0, 1, 1, 2, 3]), batch_sizes=tensor([3, 2, 1, 1]), sorted_indices=None, unsorted_indices=None)


# Zadanie 1

Zaimplementuj obiekty Dataset i DataLoader dla naszych zbiorów danych w wariantach zwracających zarówno obiekt PackedSequences, jak i prosty tensor z  wypadowanym zerami batchem.

In [7]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence, pack_sequence

class IMDBDataset(Dataset):
    def __init__(self, x, y):
        self.x = [torch.tensor(seq, dtype=torch.long) for seq in x]
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

class InsectDataset(Dataset):
    def __init__(self, X, y):
        # X: (n_samples, n_channels, n_timepoints) — aeon format
        # Zamieniamy na listę tensorów [n_timepoints, n_channels]
        self.x = [torch.tensor(X[i].T, dtype=torch.float32) for i in range(len(X))]

        classes = sorted(set(y))
        label_map = {c: i for i, c in enumerate(classes)}
        self.y = torch.tensor([label_map[yi] for yi in y], dtype=torch.long)
        self.classes = classes

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [8]:
def collate_padded(batch):
    """
    Zwraca (padded_tensor, lengths, labels).
    padded_tensor ma kształt [max_len, batch_size, features].
    """
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences])
    padded = pad_sequence(sequences, batch_first=False, padding_value=0)
    return padded, lengths, torch.stack(labels)


def collate_packed(batch):
    """
    Zwraca (PackedSequence, labels).
    enforce_sorted=False — DataLoader może losowo mieszać kolejność.
    """
    sequences, labels = zip(*batch)
    packed = pack_sequence(list(sequences), enforce_sorted=False)
    return packed, torch.stack(labels)


In [9]:
def get_imdb_dataloaders(x_train, y_train, x_test, y_test,
                         batch_size=32, variant='padded'):
    """variant: 'padded' | 'packed'"""
    train_ds = IMDBDataset(x_train, y_train)
    test_ds  = IMDBDataset(x_test,  y_test)

    collate_fn = collate_padded if variant == 'padded' else collate_packed

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, collate_fn=collate_fn)
    return train_loader, test_loader


def get_insect_dataloaders(X, y, batch_size=32, variant='padded',
                           test_ratio=0.2, seed=42):
    """variant: 'padded' | 'packed'"""
    dataset = InsectDataset(X, y)
    n = len(dataset)

    rng = np.random.default_rng(seed)
    indices = rng.permutation(n).tolist()
    split = int(n * (1 - test_ratio))

    train_ds = Subset(dataset, indices[:split])
    test_ds  = Subset(dataset, indices[split:])

    collate_fn = collate_padded if variant == 'padded' else collate_packed

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, collate_fn=collate_fn)
    return train_loader, test_loader


# Embedding

Warstwy embeddingu są elementem wykorzystywanym przy przetwarzaniu danych, gdzie elementem jest id w pewnym dyskretnym zbiorze obiektów. Przykładowo, dla danych językowych może być to zbiór możliwych słów. Warstwa przyporządkowuje każdemu id jego własny wektor, i zakładamy, że w trakcie uczenia wyuczy się podobieństwa między obiektami.

## Model LSTM

W teorii, warstwy LSTM potrafią zapamiętywać "długoterminowo" a więc to skąd wyciągamy dane nie powinno robić większego problemu. Sprawdźmy czy tak jest w rzeczywistości, implementując wersję prostego eksperymentu - implementując uczenie zarówno z wykorzystaniem obiektów PackedSequence, jak i zwyczajnych tensorów dopełnionych zerami.

Nasz model LSTM musi być przygotowany na wszystkie opisane wersje naszego eksperymentu: dane w postaci sekwencji już w przestrzeni cech i w postaci sekwencji Integerów z embeddingiem w obrębie sieci; wykorzystanie PacekdSequences lub nie.

(**UWAGA:** W przypadku wykorzystania Embeddingu i Packed Sequences jednocześnie, będzie trzeba rozpakować, embedować i spakować jeszcze raz. Obsługiwanie tych czterech wariantów w jednej klasie nie jest praktyczne, potraktuj je raczej jako ćwiczenie.)

In [10]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence

class LSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes,
                 bidirectional=False, embed=False, packed=False):
        super(LSTMNet, self).__init__()
        self.embed    = embed
        self.packed   = packed
        self.bidirectional = bidirectional

        # Gdy embed=True: input_size = rozmiar słownika (vocab_size),
        # wymiar embeddingu ustawiamy na hidden_size
        if embed:
            self.embedding = nn.Embedding(input_size, hidden_size)
            lstm_input_size = hidden_size
        else:
            lstm_input_size = input_size

        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=False,
        )

        fc_in = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_in, num_classes)

    def forward(self, x):
        if self.packed and self.embed:
            # 1) PackedSequence z integerami → rozpakuj, embeduj, spakuj ponownie
            padded, lengths = pad_packed_sequence(x, batch_first=False)
            embedded = self.embedding(padded)           # [T, B, embed_dim]
            x = pack_padded_sequence(embedded, lengths.cpu(),
                                     batch_first=False, enforce_sorted=False)
            _, (h_n, _) = self.lstm(x)

        elif self.packed:
            # 2) PackedSequence z cechami → bezpośrednio do LSTM
            _, (h_n, _) = self.lstm(x)

        elif self.embed:
            # 3) Padded tensor z integerami [T, B] → embedding → LSTM
            embedded = self.embedding(x)               # [T, B, embed_dim]
            _, (h_n, _) = self.lstm(embedded)

        else:
            # 4) Padded tensor z cechami [T, B, input_size] → LSTM
            _, (h_n, _) = self.lstm(x)

        # h_n: [num_layers * num_directions, B, hidden_size]
        if self.bidirectional:
            # ostatnia warstwa: forward = h_n[-2], backward = h_n[-1]
            last = torch.cat([h_n[-2], h_n[-1]], dim=-1)  # [B, hidden_size*2]
        else:
            last = h_n[-1]                                 # [B, hidden_size]

        return self.fc(last)


## Uczenie

Pętla ucząca dla modelu LSTM będzie analogiczna do znanych nam wcześniej

In [11]:
def validate_clf(model, loss_fn, dataloader, num_classes, device='cuda'):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    tps = torch.zeros(num_classes, device=device)
    fps = torch.zeros(num_classes, device=device)
    fns = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)   # działa dla Tensor i PackedSequence
            y = y.to(device)
            out = model(x)
            total_loss += loss_fn(out, y).item()

            preds = torch.argmax(out, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            for c in range(num_classes):
                tps[c] += ((preds == c) & (y == c)).sum()
                fps[c] += ((preds == c) & (y != c)).sum()
                fns[c] += ((preds != c) & (y == c)).sum()

    acc = correct / total
    precisions = tps / (tps + fps + 1e-8)
    recalls    = tps / (tps + fns + 1e-8)
    macro_f1   = (2 * precisions * recalls / (precisions + recalls + 1e-8)).mean().item()

    return total_loss / len(dataloader), acc, macro_f1


def fit_clf(model, optimiser, loss_fn, train_dl, val_dl,
            epochs, num_classes, device='cuda'):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x, y in train_dl:
            x = x.to(device)
            y = y.to(device)
            optimiser.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimiser.step()
            train_loss += loss.item()

        train_loss /= len(train_dl)
        val_loss, val_acc, val_f1 = validate_clf(model, loss_fn, val_dl, num_classes, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"Val F1: {val_f1:.4f}")

    return history


## Zadanie 2

Przeprowadź uczenie modelu LSTM na zadanym zbiorze i porównaj następujące podejścia:

*   LSTM, dane z paddingiem do długości najdłuższej sekwencji w batchu
*   Jak wyżej, ale Bidirectional
*   LSTM (nie bidirectional) z wykorzystaniem obiektów PaddedSequences

In [ ]:
# testy

# Zadanie 3

Dla modelu językowego, dokonaj wizualizacji embeddingów 10 przykładowych słów. Dobierz słowa samodzielnie tak, aby pokazać, że embedding częściowo (ale prwadopodobnie nie idealnie) oddają relacje semantyczne. Możesz wykorzystać gotowe metody redukcji wymiarowości np. z scikit-learn aby umieścić embeddingi w przestrzeni dwuwymiarowej.

Ponieważ zbiór pobierany z tf.keras jest już reprezentowany w postaci list liczb całkowitych, bedziesz korzystać ze słownika indeksów również dostępnego w tf.keras:

In [ ]:
word_idx = imdb.get_word_index(
    path='imdb_word_index.json'
)
print(word_idx["good"])